# 02. 研究报告评估

产出了研究报告，怎么知道它好不好？三个可自动化的维度：
- **覆盖度**：子问题是否都被回答了
- **引用准确性**：[D?] 标注的文档是否真的支持该论断
- **事实一致性**：报告内容是否与源文档矛盾（NLI 思路）

评估闭环的意义：低分维度反馈给下一轮研究，报告质量可迭代提升，而不是「一锤子买卖」。

> **检查点**：能说出「引用准确性」为什么不能靠字符串匹配评估（论断是转述，不是复制）。


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 先用 mini researcher 产出一份报告（真实 API）
import sys
sys.path.insert(0, str(ROOT := __import__('pathlib').Path.cwd()))
# 找到 agents/scripts 以便导入 researcher 的组件
while not (ROOT / 'xmake.lua').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents' / 'deep-research' / 'scripts'))
sys.path.insert(0, str(ROOT / 'agents'))

from mini_deep_researcher import CORPUS, TfidfIndex, plan_llm, synthesize_llm

question = "agent 记忆系统有哪些主流方案，如何选型"
index = TfidfIndex(CORPUS)
subs = plan_llm(client, "deepseek-v4-flash", question) if client else [question, "对比", "选型"]
notes = [(s, index.search(s, top_k=2)) for s in subs]
report = synthesize_llm(client, "deepseek-v4-flash", question, notes) if client else "toy 模式：跳过真实报告生成"
print(report)


根据材料，主流 agent 记忆系统分为两层：工作记忆（当前上下文窗口）与长期记忆（跨会话）。工作记忆通过上下文工程（compaction、隔离、caching）管理 [D7]；长期记忆方案包括 Mem0/Zep 等 [D7]。记忆与 RAG 互补：RAG 回答“世界是什么样的”，记忆回答“我们经历过什么”，生产系统两者都需要 [D4]。Reflexion 是一种从经验学习的机制，将失败反思写入情节记忆供后续检索 [D8]。选型需考虑规模与基础设施：长期记忆存储层常用 pgvector（零新增运维）、Qdrant（高性能边缘）、Milvus（十亿级分布式），Chroma 适合原型；瓶颈通常在工作记忆 [D7][D5]。

**未覆盖的方面**：具体延迟要求、存储成本量化、任务复杂度对选型的影响、不同方案的优缺点对比表格。


## 三维评估实操

下面的 cell 对每个维度做一次独立的 LLM-as-judge 调用。注意三个评审是**独立调用** —
放在同一个对话里会互相污染（评审 1 的宽松会传染给评审 2）。


In [3]:
import re

corpus_text = "\n\n".join(f"[{d['id']}] {d['text']}" for d in CORPUS)

# 维度 1：覆盖度
coverage = chat(client, f"研究问题：{question}\n子问题：{subs}\n报告：{report}\n\n每个子问题是否被回答？逐条给出 是/否，最后给覆盖度评分(0-10)。",
                system="你是苛刻的评审，只按输出格式回答。") if client else "[toy] 覆盖度评审需要 API"
print("=== 覆盖度 ===")
print(coverage)

# 维度 2：引用准确性 — 抽出报告中的 [D?] 引用逐条核对
citations = re.findall(r"\[(D\d+)\]", report)
print(f"\n=== 引用准确性（{len(citations)} 处引用）===")
if client and citations:
    verdict = chat(client, f"源文档：\n{corpus_text}\n\n报告：\n{report}\n\n报告中引用了 {sorted(set(citations))}。逐条判断：引用的文档是否支持对应论断？给出 支持/不支持/部分支持。",
                   system="你是事实核查员。只依据源文档判断，不用你自己的知识。")
    print(verdict)
else:
    print("[toy] 引用核对需要 API")

# 维度 3：事实一致性
consistency = chat(client, f"源文档：\n{corpus_text}\n\n报告：\n{report}\n\n报告中有没有与源文档矛盾的陈述？有则引用原文指出，没有则回答「无矛盾」。",
                   system="你是事实核查员。只依据源文档判断。") if client else "[toy] 一致性检查需要 API"
print("\n=== 事实一致性 ===")
print(consistency)


=== 覆盖度 ===
**逐条评估**：
- 子问题1（分类与代表方案）：**是**（但较为概括，未明确列出结构化记忆、经验回放等分类，仅隐含提及）
- 子问题2（核心机制、优缺点及应用场景）：**否**（仅对Reflexion机制有简要描述，缺乏系统对比和典型场景说明）
- 子问题3（选型权衡维度）：**是**（提到了规模、基础设施、存储层选项，但未量化延迟、成本，未系统分析任务复杂度影响）

**覆盖度评分**：4/10

=== 引用准确性（6 处引用）===


根据源文档逐条判断：

- 论断引用 [D7]：D7 明确说明工作记忆通过上下文工程管理，Mem0/Zep 管理长期记忆，支持。
- 论断引用 [D4]：D4 直接给出 RAG 与记忆的互补定义及生产系统需求，支持。
- 论断引用 [D8]：D8 描述了 Reflexion 将失败反思写入情节记忆供后续检索，支持。
- 论断引用 [D5][D7]：D5 说明存储层常用方案及选型原则，D7 指出瓶颈在工作记忆，均支持。

结论：所有引用均 **支持** 对应论断。



=== 事实一致性 ===
无矛盾


## 评估驱动迭代

三个维度中最低分的一个，就是下一轮研究的改进方向：
- 覆盖度低 → verify 环节的缺失方面作为新子问题再检索
- 引用准确性低 → synthesize 的 prompt 加强「逐句核对来源」
- 一致性低 → 检查检索是否混入了无关文档（retrieval 问题，不是 synthesis 问题）

## 练习

1. 把三个评审改成 `chat_json` 结构化输出（`{"score": int, "issues": [...]}`），写进 JSON 文件，实现多轮报告的分数追踪。
2. 故意让报告包含一句源文档没有的论断（手改 `report` 字符串），验证「事实一致性」评审能否抓住。
3. 讨论：三个评审都用同一个模型，会有系统性偏差吗？如何设计交叉验证（不同模型互评）？
